In [ ]:
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from arch import arch_model
from config import settings
from data import SQLRepository
from IPython.display import VimeoVideo
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


In [ ]:
from mock_alpha import activate_mock
activate_mock()


In [ ]:
VimeoVideo("770039650", h="c39b4b0c08", width=600)


In [ ]:
VimeoVideo("770039537", h="a20af766cc", width=600)


In [ ]:
connection = sqlite3.connect(settings.db_name, check_same_thread=False)
repo = SQLRepository(connection=connection)

print("repo type:", type(repo))
print("repo.connection type:", type(repo.connection))


In [ ]:
VimeoVideo("770039513", h="74530cf5b8", width=600)


In [ ]:
df_ambuja = repo.read_table(table_name="AMBUJACEM.BSE", limit = 2500)

print("df_ambuja type:", type(df_ambuja))
print("df_ambuja shape:", df_ambuja.shape)
df_ambuja.head()


In [ ]:
VimeoVideo("770039434", h="4fdcd5ffcb", width=600)


In [ ]:
def wrangle_data(ticker, n_observations):
    """Extract table data from database. Calculate returns.

    Parameters
    ----------
    ticker : str
        The ticker symbol of the stock (also table name in database).

    n_observations : int
        Number of observations to return.

    Returns
    -------
    pd.Series
        Name will be `"return"`. There will be no `NaN` values.
    """
    # Get table from database
    df = repo.read_table(table_name= ticker, limit = n_observations+1)

    # Sort DataFrame ascending by date
    df.sort_index(ascending=True, inplace=True)

    # Create "return" column
    df["return"] = df["close"].pct_change() * 100

    # Return returns
    return df["return"].dropna()


In [ ]:
y_ambuja = wrangle_data(ticker="AMBUJACEM.BSE", n_observations=2500)

# Is `y_ambuja` a Series?
assert isinstance(y_ambuja, pd.Series)

# Are there 2500 observations in the Series?
assert len(y_ambuja) == 2500

# Is `y_ambuja` name "return"?
assert y_ambuja.name == "return"

# Does `y_ambuja` have a DatetimeIndex?
assert isinstance(y_ambuja.index, pd.DatetimeIndex)

# Is index sorted ascending?
assert all(y_ambuja.index == y_ambuja.sort_index(ascending=True).index)

# Are there no `NaN` values?
assert y_ambuja.isnull().sum() == 0

y_ambuja.head()


In [ ]:
VimeoVideo("770039414", h="8e8317029e", width=600)


In [ ]:
y_suzlon = wrangle_data(ticker="SUZLON.BSE", n_observations=2500)

print("y_suzlon type:", type(y_suzlon))
print("y_suzlon shape:", y_suzlon.shape)
y_suzlon.head()


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot returns for `df_suzlon` and `df_ambuja`
y_suzlon.plot(ax=ax, label="SUZLON")
y_ambuja.plot(ax=ax, label="AMBUJACEM")

# Label axes
plt.xlabel("Date")
plt.ylabel("Return")

# Add legend
plt.legend();


In [ ]:
VimeoVideo("770039370", h="dde163e45b", width=600)


In [ ]:
# Create histogram of `y_ambuja`, 25 bins
plt.hist(y_ambuja, bins=25)

# Add axis labels
plt.xlabel("Daily Return")
plt.ylabel("Frequency [count]")
# Add title
plt.title("Distribution of Ambuja Cement Daily Returns");


In [ ]:
VimeoVideo("770039332", h="d43d49b8e7", width=600)


In [ ]:
suzlon_daily_volatility = y_suzlon.std()
ambuja_daily_volatility = y_ambuja.std()

print("Suzlon Daily Volatility:", suzlon_daily_volatility)
print("Ambuja Daily Volatility:", ambuja_daily_volatility)


In [ ]:
VimeoVideo("770039290", h="5b8452708a", width=600)


In [ ]:
suzlon_annual_volatility = suzlon_daily_volatility * np.sqrt(252)
ambuja_annual_volatility = ambuja_daily_volatility * np.sqrt(252)

print("Suzlon Annual Volatility:", suzlon_annual_volatility)
print("Ambuja Annual Volatility:", ambuja_annual_volatility)


In [ ]:
VimeoVideo("770039248", h="71064ba910", width=600)


In [ ]:
ambuja_rolling_50d_volatility = y_ambuja.rolling(window=50).std().dropna()

print("rolling_50d_volatility type:", type(ambuja_rolling_50d_volatility))
print("rolling_50d_volatility shape:", ambuja_rolling_50d_volatility.shape)
ambuja_rolling_50d_volatility.head()


In [ ]:
VimeoVideo("770039209", h="8250d0a2d4", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot `y_ambuja`
y_ambuja.plot(ax=ax, label="Daily Rerurn")

# Plot 'ambuja_rolling_50d_volatility'
ambuja_rolling_50d_volatility.plot(ax=ax, label="50d rolling volatility", linewidth=3)

# Add x-axis label
plt.xlabel("Date")

# Add legend
plt.legend();


In [ ]:
VimeoVideo("770039182", h="1c7ee27846", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot squared returns
(y_ambuja**2).plot(ax=ax)

# Add axis labels
plt.xlabel("Date")
plt.ylabel("Squared Return")


In [ ]:
VimeoVideo("770039152", h="74c63d13ac", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# Create ACF of squared returns
plot_acf(y_ambuja**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient");


In [ ]:
VimeoVideo("770039126", h="4cfbc287d8", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# Create ACF of squared returns
plot_pacf(y_ambuja**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient");


In [ ]:
VimeoVideo("770039107", h="8c9fbe0f4d", width=600)


In [ ]:
cutoff_test = int(len(y_ambuja) * 0.8)
y_ambuja_train = y_ambuja.iloc[:cutoff_test]

print("y_ambuja_train type:", type(y_ambuja_train))
print("y_ambuja_train shape:", y_ambuja_train.shape)
y_ambuja_train.tail()


In [ ]:
VimeoVideo("770039693", h="f06bf81928", width=600)


In [ ]:
VimeoVideo("770039053", h="beaf7753d4", width=600)


In [ ]:
# Build and train model
model = arch_model(
    y_ambuja_train,
    p=2,
    q=2,
    rescale=False
).fit(disp=0)

print("model type:", type(model))
#Show model summary
model.summary()


In [ ]:
VimeoVideo("770039014", h="5e41551d9f", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

y_ambuja_train.plot(ax=ax, label="Ambuja Daily Returns")

(2 * model.conditional_volatility).plot(
    ax=ax,
    color="C1",
    label="2 SD Conditional Volatility",
    linewidth=3
)

(-2 * model.conditional_volatility).rename("").plot(
    ax=ax,
    color="C1",
    linewidth=3
)

plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.legend();


In [ ]:
VimeoVideo("770038994", h="2a13ab49a7", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot standardized residuals
model.std_resid.plot(ax=ax, label="Standardized Residuals")

# Add axis labels
plt.xlabel("Date")

# Add legend
plt.legend();


In [ ]:
VimeoVideo("770038970", h="f76c8f6529", width=600)


In [ ]:
# Create histogram of standardized residuals, 25 bins
plt.hist(model.std_resid, bins=25)

# Add axis labels
plt.xlabel("Standardized Residual")
plt.ylabel("Frequency [count]")

# Add title
plt.title("Distribution of Standardized Residuals");


In [ ]:
VimeoVideo("770038952", h="c7a3cfe34f", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_acf(model.std_resid**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient");


In [ ]:
VimeoVideo("770038921", h="f74869b8fc", width=600)


In [ ]:
one_day_forecast = model.forecast(horizon=1, reindex=False).variance

print("one_day_forecast type:", type(one_day_forecast))
one_day_forecast


In [ ]:
VimeoVideo("770038861", h="10efe8c445", width=600)


In [ ]:
# Create empty list to hold predictions
predictions = []

# Calculate size of test data (20%)
test_size = int(len(y_ambuja) * 0.2)

# Walk forward
for i in range(test_size):
    # Create test data
    y_train = y_ambuja.iloc[: -(test_size - i)]

    # Train model
    model = arch_model(y_train, p=1, q=1, rescale=False).fit(disp=0)


    # Generate next prediction (volatility, not variance)
    next_pred = model.forecast(horizon=1, reindex=False).variance.iloc[0, 0] ** 0.5

    # Append prediction to list
    predictions.append(next_pred)

# Create Series from predictions list
y_test_wfv = pd.Series(predictions, index=y_ambuja.tail(test_size).index)

print("y_test_wfv type:", type(y_test_wfv))
print("y_test_wfv shape:", y_test_wfv.shape)
y_test_wfv.head()


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot returns for test data
y_ambuja.tail(test_size).plot(ax=ax, label="Ambuja Return")

# Plot volatility predictions * 2
(2 * y_test_wfv).plot(ax=ax, c="C1", label="2 SD Predicted Volatility")

# Plot volatility predictions * -2
(-2 * y_test_wfv).plot(ax=ax, c="C1")

# Label axes
plt.xlabel("Date")
plt.ylabel("Return")

# Add legend
plt.legend();


In [ ]:
VimeoVideo("770038804", h="8976257596", width=600)


In [ ]:
# Generate 5-day volatility forecast
prediction = model.forecast(horizon=5, reindex=False).variance ** 0.5
print(prediction)

# Calculate forecast start date
start = prediction.index[0] + pd.DateOffset(days=1)

# Create date range
prediction_dates = pd.bdate_range(start=start,periods=prediction.shape[1])

# Create prediction index labels, ISO 8601 format
prediction_index = [d.isoformat() for d in prediction_dates]

print("prediction_index type:", type(prediction_index))
print("prediction_index len:", len(prediction_index))
prediction_index[:3]


In [ ]:
VimeoVideo("770039565", h="d419d0a78d", width=600)


In [ ]:
def clean_prediction(prediction):
    """Reformat model prediction to JSON.

    Parameters
    ----------
    prediction : pd.DataFrame
        Variance from a `ARCHModelForecast`

    Returns
    -------
    dict
        Forecast of volatility. Each key is date in ISO 8601 format.
        Each value is predicted volatility.
    """
    # Calculate forecast start date
    # Calculate forecast start date
    start = prediction.index[0] + pd.DateOffset(days=1)

    # Create date range
    prediction_dates = pd.bdate_range(start=start,periods=prediction.shape[1])

    # Create prediction index labels, ISO 8601 format
    prediction_index = [d.isoformat() for d in prediction_dates]

    # Extract predictions from DataFrame, get square root
    data = prediction.values.flatten() ** 0.5

    # Combine `data` and `prediction_index` into Series
    prediction_formatted = pd.Series(data, index=prediction_index)

    # Return Series as dictionary
    return prediction_formatted.to_dict()


In [ ]:
prediction = model.forecast(horizon=10, reindex=False).variance
prediction_formatted = clean_prediction(prediction)

# Is `prediction_formatted` a dictionary?
assert isinstance(prediction_formatted, dict)

# Are keys correct data type?
assert all(isinstance(k, str) for k in prediction_formatted.keys())

# Are values correct data type
assert all(isinstance(v, float) for v in prediction_formatted.values())

prediction_formatted
